In [2]:
import tarfile
import os 
import json
import re
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import time

# Machine Learning 
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report,
    f1_score,
    precision_recall_fscore_support,
    roc_curve,
    auc,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Plotting Libraries
import matplotlib.pyplot as plt

class TweetFeatureExtractor:
    def __init__(self, window_hours=4.0):
        # Convert window parameter to seconds 
        self.window_limit_secs = window_hours * 3600
        
        # Stream B: Catch explicit skepticism/denial words on Twitter
        self.doubt_pattern = re.compile(
            r'\b('
            r'fake|hoax|debunk|false|unverified|misleading|contradict|bullshit|bs|lie|lies|fabricated|forged|deceptive|untrue|'
            r'source|sources|prove|proof|verification|verify|confirm|confirmation|check|factcheck|authentic|reliable|reputable|'
            r'speculate|speculation|alleged|allegedly|reportedly|apparently|seems|suspect|unconfirmed|supposedly|rumor|rumour|rumors|rumours|'
            r'really|seriously|actual|actually|legit|hoax|real\?'
            r')\b',
            re.IGNORECASE
        )

    def _parse_twitter_time(self, ts):
        # Standard Twitter API metadata timestamp layout
        return datetime.strptime(ts, '%a %b %d %H:%M:%S +0000 %Y')
        
    def extract_thread_features(self, thread_dir):
        thread_path = Path(thread_dir)
        source_dir = thread_path / "source-tweet"
        reactions_dir = thread_path / "reactions"

        # Grab the source tweet 
        source_file = next(source_dir.glob("*.json"))
        with open(source_file, 'r', encoding='utf-8') as f:
            source_data = json.load(f)
            
        source_text = source_data.get('text', '')
        t_source = self._parse_twitter_time(source_data['created_at'])
        
        replies = []
        time_deltas = [] 
        
        if reactions_dir.exists():
            for reply_file in reactions_dir.glob('*.json'):
                try:
                    with open(reply_file, 'r', encoding='utf-8') as f:
                        reply_data = json.load(f)
                    
                    t_reply = self._parse_twitter_time(reply_data['created_at'])
                    elapsed_secs = (t_reply - t_source).total_seconds()
                    
                    # Enforce our strict pre-viral window constraint
                    if 0 <= elapsed_secs <= self.window_limit_secs:
                        replies.append(reply_data.get('text', ''))
                        time_deltas.append(elapsed_secs / 3600.0)
                except (json.JSONDecodeError, KeyError, ValueError):
                    # Skip anomaly files
                    continue

        # Stream B: Analyse skepticism text metrics
        all_replies_text = " ".join(replies)
        words = all_replies_text.split()
        word_count = len(words)
        
        punc_density = 0.0
        doubt_freq = 0.0
        
        if word_count > 0:
            # Count expressive punctuation marks relative to total words
            total_punc = all_replies_text.count('?') + all_replies_text.count('!')
            punc_density = total_punc / word_count
            
            matches = self.doubt_pattern.findall(all_replies_text)
            doubt_freq = len(matches) / word_count

        # Stream C: Track conversation kinetic growth 
        reply_count = len(replies)
        velocity = 0.0
        
        if reply_count > 0 and time_deltas:
            time_deltas.sort()
            total_window_span = time_deltas[-1] - time_deltas[0]
            
            # Rate of response influx per hour across active window duration
            if total_window_span > 0:
                velocity = reply_count / total_window_span
            else:
                # 1 second to capture burst of reply
                epsilon_hours = 0.000277
                velocity = reply_count / epsilon_hours 

        return {
            "source_text_raw": source_text, 
            "punc_density": round(punc_density, 5),
            "doubt_freq": round(doubt_freq, 5),
            "reply_count": reply_count,
            "reply_velocity_per_hr": round(velocity, 2)
        }

dataset_folder = 'pheme-rnr-dataset'

# Safe extraction checkpoint at root level
if not os.path.exists(dataset_folder):
    with tarfile.open("phemernrdataset.tar.bz2", "r:bz2") as files:
        files.extractall('.')
    print("Extraction complete!")

# Start operational execution clock
start_time = time.time()
extractor = TweetFeatureExtractor()
all_features = []
dataset_path = Path(dataset_folder)

if dataset_path.exists():
    for event_dir in dataset_path.iterdir():
        if not event_dir.is_dir():
            continue
            
        for label in ['rumours', 'non-rumours']:
            label_dir = event_dir / label
            if not label_dir.exists():
                continue
                
            for thread_dir in label_dir.iterdir():
                if not thread_dir.is_dir():
                    continue
                    
                try:
                    # Execute extraction over targeted subfolders
                    features = extractor.extract_thread_features(thread_dir)
                    features.update({
                        'thread_id': thread_dir.name,
                        'event': event_dir.name,
                        'is_rumour': 1 if label == 'rumours' else 0
                    })
                    all_features.append(features)
                except (json.JSONDecodeError, KeyError, ValueError, StopIteration):
                    # skip anomaly/missing files
                    continue

if all_features:
    df = pd.DataFrame(all_features)
    output_file = "pheme_extracted_features.csv"
    df.to_csv(output_file, index=False)
    
    elapsed = time.time() - start_time
    print(f"\nDone! Processed {len(df)} threads in {elapsed:.2f}s.")
    print(f"Saved to '{output_file}'")

    # Feature Preparation
    # Stream A: Corpus-wide Text Vectorization with SelectKBest
    tfidf = TfidfVectorizer(max_features=500, stop_words='english')
    X_tfidf_raw = tfidf.fit_transform(df['source_text_raw']).toarray()

    # Retain top 50 relevant textual features using SelectKBest
    selector = SelectKBest(score_func=f_classif, k=min(50, X_tfidf_raw.shape[1]))
    X_tfidf_selected = selector.fit_transform(X_tfidf_raw, df['is_rumour'])

    token_indices = selector.get_support(indices=True)
    all_tokens = np.array(tfidf.get_feature_names_out())
    selected_tokens = all_tokens[token_indices]

    df_tfidf = pd.DataFrame(X_tfidf_selected, columns=[f"tfidf_{token}" for token in selected_tokens])

    # Streams B & C: Numerical Features Scaling
    numerical_features = ['punc_density', 'doubt_freq', 'reply_count', 'reply_velocity_per_hr']
    scaler = StandardScaler()
    X_numerical_scaled = scaler.fit_transform(df[numerical_features].values)
    df_numerical = pd.DataFrame(X_numerical_scaled, columns=numerical_features)

    # All Features & Text Only
    X_full = pd.concat([df_tfidf, df_numerical], axis=1)
    X_baseline = df_tfidf.copy()

    events = df['event'].str.lower().values
    y_all = df['is_rumour'].values

    # Train/Test Split 
    is_test = np.array(['sydney' in e for e in events])
    is_train = ~is_test

    # Feature matrices for RF (text-only) and RF (All features)
    X_train_base, X_test_base = X_baseline[is_train], X_baseline[is_test]
    X_train_full, X_test_full = X_full[is_train], X_full[is_test]
    y_train, y_test = y_all[is_train], y_all[is_test]

    print(f"Training Samples: {len(X_train_full)}")
    print(f"Testing Samples (Unseen Event - Sydney Siege): {len(X_test_full)}")

    # Model Evaluation & Plotting
    models = {
        "RF (Text Only)": (RandomForestClassifier(n_estimators=100, random_state=42), X_train_base, X_test_base),
        "RF (All Features)": (RandomForestClassifier(n_estimators=100, random_state=42), X_train_full, X_test_full),
        "SVM (Full Features)": (SVC(probability=True, random_state=42), X_train_full, X_test_full)
    }

    results = []
    fig_roc, ax_roc = plt.subplots(figsize=(8, 6))

    for name, (clf, X_tr, X_te) in models.items():
        clf.fit(X_tr, y_train)
        y_pred = clf.predict(X_te)

        # Get probabilities for ROC curve
        if hasattr(clf, "predict_proba"):
            y_proba = clf.predict_proba(X_te)[:, 1]
        else:
            y_proba = clf.decision_function(X_te)

        # Record metrics
        prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='macro')
        acc = (y_pred == y_test).mean()

        results.append({
            "Model Architecture": name,
            "Accuracy": round(acc, 4),
            "Macro Precision": round(prec, 4),
            "Macro Recall": round(rec, 4),
            "Macro F1-Score": round(f1, 4)
        })

        # Plot ROC curve for current model 
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        roc_auc = auc(fpr, tpr)
        ax_roc.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc:.3f})")

        # Generate confusion matrix and feature importances for the RF (All Features)
        if name == "RF (All Features)":
            print("\n Classification Report for RF (All Features):")
            print(classification_report(y_test, y_pred, target_names=['Non-Rumour', 'Rumour']))

            # Confusion Matrix
            cm = confusion_matrix(y_test, y_pred)
            disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Non-Rumour', 'Rumour'])
            fig_cm, ax_cm = plt.subplots(figsize=(6, 5))
            disp.plot(ax=ax_cm, cmap=plt.cm.Blues)
            ax_cm.set_title("Confusion Matrix: RF (All Features) (Sydney Siege)")
            plt.tight_layout()
            plt.savefig("confusion.png") 
            plt.close(fig_cm)

            # Feature Importance Plot
            importances = clf.feature_importances_
            feature_names = X_tr.columns
            indices = np.argsort(importances)[-15:]  # Top 15 features

            fig_fi, ax_fi = plt.subplots(figsize=(8, 6))
            ax_fi.barh(range(len(indices)), importances[indices], align='center', color='skyblue')
            ax_fi.set_yticks(range(len(indices)))
            ax_fi.set_yticklabels([feature_names[i] for i in indices])
            ax_fi.set_xlabel("Relative Feature Importance")
            ax_fi.set_title("Top Predictive Features (RF (All Features))")
            plt.tight_layout()
            plt.savefig("feature_importance.png")
            plt.close(fig_fi)

    # ROC Curve plot
    ax_roc.plot([0, 1], [0, 1], 'k--', label="Random Classifier")
    ax_roc.set_xlabel("False Positive Rate")
    ax_roc.set_ylabel("True Positive Rate")
    ax_roc.set_title("ROC Curves Comparison (Tested on Sydney Siege)")
    ax_roc.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig("roc_curves.png")
    plt.close(fig_roc)
    
    # Output Evaluation Summary Table
    results_df = pd.DataFrame(results)
    print("Experiment Evaluation Summary")
    print(results_df.to_string(index=False))

    # Trend Emergence Score
    print("\n Trend Emergence Scores for Test Data")
    rf = models["RF (All Features)"][0]
    rf.fit(X_train_full, y_train)

    p_rumour = rf.predict_proba(X_test_full)[:, 1]
    test_indices = np.where(is_test)[0]
    df_test_raw = df.iloc[test_indices].copy()

    # Extract kinetic features
    velocity_hrs = np.log1p(df_test_raw['reply_velocity_per_hr'].values)
    punc_density = df_test_raw['punc_density'].values
    doubt_freq = df_test_raw['doubt_freq'].values
    
    # Normalize velocity using Min-Max scaling 
    vel_min, vel_max = velocity_hrs.min(), velocity_hrs.max()
    norm_velocity = (velocity_hrs - vel_min) / (vel_max - vel_min + 1e-6)
    
    # Normalize emotional/doubt intensity (Stream B)
    emotional_intensity = (punc_density + doubt_freq) / 2.0
    norm_emotion = (emotional_intensity - emotional_intensity.min()) / (emotional_intensity.max() - emotional_intensity.min() + 1e-6)
    
    # Compute Trend Emergence Score 
    # Weights: 50% Rumour Probability, 35% Velocity Acceleration, 15% Skepticism/Emotion
    w_rumour = 0.50
    w_velocity = 0.35
    w_emotion = 0.15
    
    trend_emergence_score = (
        (w_rumour * p_rumour) + 
        (w_velocity * norm_velocity) + 
        (w_emotion * norm_emotion)
    )
    
    # Append Scores to Test DataFrame
    df_test_raw['rumour_probability'] = np.round(p_rumour, 4)
    df_test_raw['trend_emergence_score'] = np.round(trend_emergence_score, 4)
    
    # Define Alert Triage Threshold (e.g., Score > 0.65 triggers a Pre-Viral Alert)
    TRIAGE_THRESHOLD = 0.65
    df_test_raw['triage_alert'] = df_test_raw['trend_emergence_score'] >= TRIAGE_THRESHOLD
    
    # Display Top High-Risk Pre-Viral Threats
    high_risk_threads = df_test_raw[df_test_raw['triage_alert']].sort_values(
        by='trend_emergence_score', ascending=False
    )
    
    print(f"Total Test Threads Evaluated (Sydney Siege): {len(df_test_raw)}")
    print(f"Pre-Viral Alerts Triggered (Score >= {TRIAGE_THRESHOLD}): {len(high_risk_threads)}")
    print("\nTop 5 High-Risk Threads Flagged for Immediate Triage:")
    print(high_risk_threads[['thread_id', 'rumour_probability', 'reply_velocity_per_hr', 'trend_emergence_score', 'is_rumour']].head())
    
else:
    print("Extraction failed.")


Done! Processed 5802 threads in 29.35s.
Saved to 'pheme_extracted_features.csv'
Training Samples: 4581
Testing Samples (Unseen Event - Sydney Siege): 1221

 Classification Report for RF (All Features):
              precision    recall  f1-score   support

  Non-Rumour       0.73      0.83      0.77       699
      Rumour       0.72      0.58      0.64       522

    accuracy                           0.72      1221
   macro avg       0.72      0.71      0.71      1221
weighted avg       0.72      0.72      0.72      1221

Experiment Evaluation Summary
 Model Architecture  Accuracy  Macro Precision  Macro Recall  Macro F1-Score
     RF (Text Only)    0.6880           0.6808        0.6753          0.6769
  RF (All Features)    0.7232           0.7217        0.7054          0.7084
SVM (Full Features)    0.6626           0.7008        0.6177          0.5987

 Trend Emergence Scores for Test Data
Total Test Threads Evaluated (Sydney Siege): 1221
Pre-Viral Alerts Triggered (Score >= 0.65):